In [1]:
# ============================================
# Exportar crops (infected / uninfected) usando ANOTAÇÕES EM JPG
# - Aceita jpg com pontos coloridos (vermelho/amarelo); qualquer pixel não-preto é "infectado"
# - Se houver máscara de células: rotula célula a célula (igual ao pipeline NIfTI)
# - Se não houver máscara: exporta "annotation-only" (positivos = blobs da anotação; negativos amostrados)
# - Compatível com NumPy 2.0
# ============================================

from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import imageio.v2 as iio
import cv2
from skimage import measure
from tqdm import tqdm

# -------------------- HELPERS --------------------
def load_img_rgb(path):
    return np.array(Image.open(path).convert("RGB"), dtype=np.uint8)

def load_gray(path):
    return np.array(Image.open(path).convert("L"), dtype=np.uint8)

def load_mask_ids(path):
    m = iio.imread(path)
    if m.ndim == 3:
        m = m[..., 0]
    return m.astype(np.int32)

def _sanitize_u8(arr):
    if arr is None: return None
    if arr.dtype == np.void or arr.dtype == object:
        try: arr = np.array(arr.tolist())
        except Exception: arr = np.asarray(arr)
    if not np.issubdtype(arr.dtype, np.integer):
        arr = np.asarray(arr, dtype=np.float32)
        arr = np.clip(arr, 0, 255)
    return arr.astype(np.uint8, copy=False)

def _resize_u8_cv(img_u8, out_wh, interp=cv2.INTER_AREA):
    img_u8 = _sanitize_u8(img_u8)
    ow, oh = out_wh  # (width, height)
    return cv2.resize(img_u8, (ow, oh), interpolation=interp).astype(np.uint8, copy=False)

def _resize_bin_cv(bin_u8, out_wh):
    return _resize_u8_cv(bin_u8, out_wh, interp=cv2.INTER_NEAREST)

def _crop_with_padding(img, bbox, pad=6):
    y0, x0, y1, x1 = bbox
    H, W = img.shape[:2]
    y0 = max(y0 - pad, 0); x0 = max(x0 - pad, 0)
    y1 = min(y1 + pad, H); x1 = min(x1 + pad, W)
    return img[y0:y1, x0:x1]

# -------------------- TRANSFORM CANDIDATES --------------------
def _candidates(a2d):
    return [
        ("id",           a2d),
        ("rot90",        np.rot90(a2d, 1)),
        ("rot180",       np.rot90(a2d, 2)),
        ("rot270",       np.rot90(a2d, 3)),
        ("transpose",    a2d.T),
        ("flipud",       np.flipud(a2d)),
        ("fliplr",       np.fliplr(a2d)),
        ("rot90+fliplr", np.fliplr(np.rot90(a2d, 1))),
        ("rot90+flipud", np.flipud(np.rot90(a2d, 1))),
    ]

# -------------------- JPG → MAPA BINÁRIO --------------------
def _infected_map_from_jpg(jpg_path, target_shape, cell_union=None,
                           force_transform=None, dilate_px=2, non_black_thr=20):
    """
    Converte anotação JPG para mapa binário.
    - non_black_thr: valor de corte (0..255). Qualquer pixel com max(R,G,B) > thr é "infectado".
    - Se cell_union for dado, escolhe a melhor transformação (rot/flip/transpose) por interseção.
    """
    H, W = target_shape
    ann = load_img_rgb(jpg_path)  # (Hann, Wann, 3)
    # binário por "não preto"
    mask = (ann.max(axis=-1) > non_black_thr).astype(np.uint8)

    # resize ao target
    if mask.shape != (H, W):
        mask = _resize_bin_cv(mask, (W, H))

    # escolhe transformação (se houver célula)
    if cell_union is not None:
        best = (-1, "id", mask)
        cand = _candidates(mask)
        if force_transform is not None:
            cand = [(n, a) for n, a in cand if n == force_transform] or [("id", mask)]
        for name, mm in cand:
            m2 = mm
            if m2.shape != (H, W):
                m2 = _resize_bin_cv(m2, (W, H))
            inter = int(np.count_nonzero((m2 == 1) & (cell_union == 1)))
            if inter > best[0]:
                best = (inter, name, m2)
        score, tr, mask = best
    else:
        tr, score = "id", int(mask.sum())

    # dilata para melhorar cobertura
    if dilate_px and dilate_px > 0:
        k = 2*dilate_px + 1
        mask = cv2.dilate(mask, np.ones((k, k), np.uint8))

    status = "ok" if mask.max() > 0 else "empty"
    return mask.astype(np.uint8), tr, score, status

# -------------------- PIPELINE --------------------
def export_from_jpg_annotations(
    img_dir, ann_jpg_dir, out_dir,
    mask_raw_dir=None,           # opcional
    split_name="train",
    resize_to=(224, 224),
    frac_thresh=0.10,
    force_transform=None,
    dilate_px=2,
    min_blob_area=25,            # para ignorar ruído no modo "sem máscara"
    neg_per_img=3,               # nº de negativos no modo "sem máscara"
    skip_existing=True,
    non_black_thr=20,
):
    """
    - Se mask_raw_dir existir e houver máscara para a imagem -> classifica célula a célula.
    - Caso contrário -> usa "annotation-only": blobs positivos + amostragem de negativos fora da anotação.
    """
    img_dir      = Path(img_dir)
    ann_jpg_dir  = Path(ann_jpg_dir)
    out_dir      = Path(out_dir)
    mask_raw_dir = Path(mask_raw_dir) if mask_raw_dir else None

    out_pos = out_dir / split_name / "infected"
    out_neg = out_dir / split_name / "uninfected"
    out_pos.mkdir(parents=True, exist_ok=True)
    out_neg.mkdir(parents=True, exist_ok=True)

    imgs = sorted(img_dir.glob("*.bmp"))
    print(f"{len(imgs)} imagens em {img_dir}")

    records, decisions = [], []
    miss_ann, miss_mask = [], []
    used_mode = {"cellwise":0, "anno_only":0}

    for img_path in tqdm(imgs):
        stem = img_path.stem
        ann_path = ann_jpg_dir / f"{stem}.jpg"
        if not ann_path.exists():
            # tenta .png como fallback
            png_alt = ann_jpg_dir / f"{stem}.png"
            if png_alt.exists(): ann_path = png_alt
        if not ann_path.exists():
            miss_ann.append(stem); continue

        img = _sanitize_u8(load_img_rgb(img_path))
        H, W = img.shape[:2]

        # tenta máscara de células (se fornecida)
        mask_path = None
        if mask_raw_dir:
            cand = list(mask_raw_dir.glob(f"{stem}*raw*.png")) or list(mask_raw_dir.glob(f"{stem}*.png"))
            if cand: mask_path = cand[0]

        if mask_path and mask_path.exists():
            # ----- modo 1: célula a célula -----
            cell_mask = load_mask_ids(mask_path)
            if np.max(cell_mask) == 0:
                miss_mask.append(stem);  # máscara vazia → cai para anno-only
                cell_mask = None

            cell_union = (cell_mask > 0).astype(np.uint8) if cell_mask is not None else None

            ann_bin, tr, score, status = _infected_map_from_jpg(
                ann_path, (H, W), cell_union=cell_union,
                force_transform=force_transform, dilate_px=dilate_px,
                non_black_thr=non_black_thr
            )

            decisions.append({
                "image": stem, "transform": tr, "align_score": int(score),
                "mode": "cellwise", "status": status
            })

            if cell_mask is not None:
                used_mode["cellwise"] += 1
                for r in measure.regionprops(cell_mask):
                    rid = r.label
                    region = (cell_mask == rid)
                    inter = np.count_nonzero(region & (ann_bin > 0))
                    frac_on_cell = inter / max(region.sum(), 1)

                    y0, x0, y1, x1 = r.bbox
                    crop = _crop_with_padding(img, (y0, x0, y1, x1), pad=6)
                    if crop.size == 0 or crop.shape[0] == 0 or crop.shape[1] == 0:
                        continue

                    try:
                        crop = _resize_u8_cv(crop, resize_to)
                    except Exception as e:
                        print(f"[WARN] resize falhou em {stem} cell {rid}: dtype={crop.dtype}, shape={crop.shape} | {e}")
                        continue

                    label = 1 if frac_on_cell >= frac_thresh else 0
                    out_path = (out_pos if label == 1 else out_neg) / f"{stem}_cell{rid:04d}.png"
                    if not (skip_existing and out_path.exists()):
                        iio.imwrite(out_path, crop)

                    records.append({
                        "split": split_name,
                        "image": stem,
                        "cell_id": int(rid),
                        "bbox": (int(y0), int(x0), int(y1), int(x1)),
                        "frac_on_cell": float(frac_on_cell),
                        "label": int(label),
                        "path": str(out_path),
                        "transform": str(tr),
                        "mode": "cellwise",
                        "status": status,
                    })
                continue  # próxima imagem

        # ----- modo 2: annotation-only (sem máscara) -----
        used_mode["anno_only"] += 1
        cell_union = None  # não há células
        ann_bin, tr, score, status = _infected_map_from_jpg(
            ann_path, (H, W), cell_union=None,
            force_transform=force_transform, dilate_px=dilate_px,
            non_black_thr=non_black_thr
        )
        decisions.append({
            "image": stem, "transform": tr, "align_score": int(score),
            "mode": "anno_only", "status": status
        })

        # blobs positivos
        lab = measure.label(ann_bin > 0, connectivity=2)
        props = [r for r in measure.regionprops(lab) if r.area >= min_blob_area]
        # positivos
        for i, r in enumerate(props, 1):
            cy, cx = map(int, r.centroid)
            half = resize_to[1] // 2  # assume quadrado (224)
            y0, y1 = max(0, cy-half), min(H, cy+half)
            x0, x1 = max(0, cx-half), min(W, cx+half)
            crop = img[y0:y1, x0:x1]
            crop = cv2.resize(crop, resize_to, interpolation=cv2.INTER_CUBIC)
            out_path = out_pos / f"{stem}_inf_{i:03d}.png"
            if not (skip_existing and out_path.exists()):
                iio.imwrite(out_path, crop)
            records.append({
                "split": split_name, "image": stem,
                "cell_id": int(i),
                "bbox": (int(y0), int(x0), int(y1), int(x1)),
                "frac_on_cell": 1.0,
                "label": 1,
                "path": str(out_path),
                "transform": str(tr),
                "mode": "anno_only",
                "status": status,
            })

        # negativos (amostra longe da anotação)
        safe = (ann_bin == 0).astype(np.uint8)
        for j in range(neg_per_img):
            tries = 0
            while tries < 50:
                cy = np.random.randint(0, H)
                cx = np.random.randint(0, W)
                rr = 18
                y0, y1 = max(0, cy-rr), min(H, cy+rr)
                x0, x1 = max(0, cx-rr), min(W, cx+rr)
                if safe[y0:y1, x0:x1].mean() > 0.99:
                    half = resize_to[1]//2
                    y0, y1 = max(0, cy-half), min(H, cy+half)
                    x0, x1 = max(0, cx-half), min(W, cx+half)
                    crop = img[y0:y1, x0:x1]
                    crop = cv2.resize(crop, resize_to, interpolation=cv2.INTER_CUBIC)
                    out_path = out_neg / f"{stem}_neg_{j:02d}.png"
                    if not (skip_existing and out_path.exists()):
                        iio.imwrite(out_path, crop)
                    records.append({
                        "split": split_name, "image": stem,
                        "cell_id": int(10000+j),  # id sintético
                        "bbox": (int(y0), int(x0), int(y1), int(x1)),
                        "frac_on_cell": 0.0,
                        "label": 0,
                        "path": str(out_path),
                        "transform": str(tr),
                        "mode": "anno_only",
                        "status": status,
                    })
                    break
                tries += 1

    # --- salvar metadados ---
    out_dir.mkdir(parents=True, exist_ok=True)
    df  = pd.DataFrame(records)
    dec = pd.DataFrame(decisions)
    (out_dir / f"cells_{split_name}.csv").write_text(df.to_csv(index=False))
    (out_dir / f"alignment_{split_name}.csv").write_text(dec.to_csv(index=False))

    print("Resumo classes:", df['label'].value_counts(dropna=False).to_dict() if len(df) else "sem registros")
    if miss_ann:  print(f"Sem anotação JPG: {len(miss_ann)} (ex.: {miss_ann[:3]})")
    if miss_mask: print(f"Máscara vazia/ausente em {len(miss_mask)} imagens (caiu para anno_only).")
    print("Modos usados → célula-a-célula:", used_mode["cellwise"], "| annotation-only:", used_mode["anno_only"])
    return df, dec


In [2]:
df_train, dec_train = export_from_jpg_annotations(
    img_dir      = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\train",
    ann_jpg_dir  = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\annot_dir_dani\train",  # suas JPGs
    mask_raw_dir = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir_masks\train_masks\raw",  # opcional; se não tiver, passe None
    out_dir      = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir_annot_jpg",
    split_name   = "train",
    resize_to    = (224, 224),
    frac_thresh  = 0.10,       # célula marcada se >=10% coberta por anotação
    force_transform=None,      # ou "transpose" etc. se souber que é fixo
    dilate_px    = 2,          # aumenta cobertura dos pontos
    min_blob_area= 25,
    neg_per_img  = 3,
    non_black_thr= 20          # >0 já funciona, 20 evita ruído de compressão
)


221 imagens em C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\train


100%|██████████| 221/221 [00:57<00:00,  3.84it/s]

Resumo classes: {0: 3856, 1: 109}
Modos usados → célula-a-célula: 221 | annotation-only: 0


In [3]:
df_val, dec_val = export_from_jpg_annotations(
    img_dir      = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\val",
    ann_jpg_dir  = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\annot_dir_dani\val",  # suas JPGs
    mask_raw_dir = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir_masks\val_masks\raw",  # opcional; se não tiver, passe None
    out_dir      = r"C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir_annot_jpg",
    split_name   = "val",
    resize_to    = (224, 224),
    frac_thresh  = 0.10,       # célula marcada se >=10% coberta por anotação
    force_transform=None,      # ou "transpose" etc. se souber que é fixo
    dilate_px    = 2,          # aumenta cobertura dos pontos
    min_blob_area= 25,
    neg_per_img  = 3,
    non_black_thr= 20          # >0 já funciona, 20 evita ruído de compressão
)


221 imagens em C:\Users\sthem\OneDrive\Documentos\GitHub\master-2025\3-Datasets\img-lcm\img_dir\val


100%|██████████| 221/221 [01:03<00:00,  3.49it/s]

Resumo classes: {0: 3687, 1: 124}
Modos usados → célula-a-célula: 221 | annotation-only: 0
